In [2]:
import pandas as pd
import ast

In [3]:
movies = pd.read_csv('../data/raw/tmdb_5000_movies.csv')
credits = pd.read_csv('../data/raw/tmdb_5000_credits.csv')

print(movies.shape, credits.shape)

(4803, 20) (4803, 4)


In [4]:
movies = movies.merge(credits, left_on='id', right_on='movie_id')
print(movies.columns.tolist())

['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language', 'original_title', 'overview', 'popularity', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title_x', 'vote_average', 'vote_count', 'movie_id', 'title_y', 'cast', 'crew']


In [5]:
movies.head(2)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,...,spoken_languages,status,tagline,title_x,vote_average,vote_count,movie_id,title_y,cast,crew
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...",...,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...",...,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."


In [6]:
movies = movies[
    ["movie_id", "title_x", "overview", "genres", "keywords", "cast", "crew"]
]
movies.rename(columns={"title_x": "title"}, inplace=True)
movies.head(2)


,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."


In [7]:
def parse_list(obj):
    try:
        return [i["name"] for i in ast.literal_eval(obj)]
    except Exception:
        return []


def get_director(obj):
    try:
        for i in ast.literal_eval(obj):
            if i["job"] == "Director":
                return [i["name"].replace(" ", "")]
        return []
    except Exception:
        return []


def top_cast(obj, n=8):
    try:
        return [i["name"].replace(" ", "") for i in ast.literal_eval(obj)[:n]]
    except Exception:
        return []

In [8]:
movies["genres"] = movies["genres"].apply(parse_list)
movies["keywords"] = movies["keywords"].apply(parse_list)
movies["cast"] = movies["cast"].apply(top_cast)
movies["crew"] = movies["crew"].apply(get_director)

movies.head(2)


,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[SamWorthington, ZoeSaldana, SigourneyWeaver, ...",[JamesCameron]
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...","[JohnnyDepp, OrlandoBloom, KeiraKnightley, Ste...",[GoreVerbinski]


In [9]:
movies["tags"] = (
    movies["overview"].fillna("").apply(lambda x: x.split())
    + movies["genres"]
    + movies["keywords"]
    + movies["cast"]
    + movies["crew"]
)

movies[["title", "tags"]].head(2)


,title,tags
0,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d..."


In [10]:
movies = movies[["movie_id", "title", "tags"]]
movies["tags"] = movies["tags"].apply(lambda x: " ".join(x).lower())

movies.head(2)


,movie_id,title,tags
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha..."


In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf = TfidfVectorizer(max_features=5000, stop_words="english")
matrix = tfidf.fit_transform(movies["tags"])

similarity = cosine_similarity(matrix)
print(similarity.shape)


(4803, 4803)


In [12]:
def recommend(title):
    idx = movies[movies["title"] == title].index[0]
    distances = sorted(
        list(enumerate(similarity[idx])), reverse=True, key=lambda x: x[1]
    )
    for i in distances[1:21]:
        print(movies.iloc[i[0]].title)


recommend("Avatar")


Aliens
Moonraker
Alien³
Alien
Silent Running
Spaceballs
Lost in Space
Lockout
Mission to Mars
Lifeforce
Planet of the Apes
Galaxina
Starship Troopers
Cargo
Treasure Planet
2001: A Space Odyssey
Home
Dune
Space Pirate Captain Harlock
Gravity


In [ ]:
import os

os.makedirs("../data/processed", exist_ok=True)
movies.to_csv("../data/processed/movies_processed.csv", index=False)
print("Saved!")

Saved!
